# 02 — Baselines Clássicos (Naive Bayes e SVM)**TCC: Detecção de Smishing em Idosos com Modelos de Linguagem Natural**Estabelece o piso de desempenho, conforme a seção 4.4.3. Se BERTimbau e Llamanão superarem estes modelos de forma relevante, a complexidade adicional não sejustifica — e isso é um achado, não um fracasso.| Modelo | Implementação | Justificativa ||---|---|---|| Complement Naive Bayes | `sklearn.naive_bayes.ComplementNB` | Variante do Naive Bayes otimizada para classes desbalanceadas || SVM linear | `LinearSVC` + `CalibratedClassifierCV` | Forte baseline para texto; a calibração fornece as pontuações |**Pré-processamento:** NLTK e spaCy, conforme as seções 4.3.2.1 e 4.3.2.2, maissubstituição de URLs, telefones e valores por tokens especiais — o que preservaa *presença* dessas entidades como sinal.> O vectorizador TF-IDF é ajustado **somente no treino** e depois aplicado a val> e teste. Ajustá-lo no corpus completo seria vazamento.

## 1. Setup

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────
# No Colab CADA notebook roda em um runtime próprio: instalar dependências
# em um notebook não vale para os outros. Por isso esta célula se repete em
# todos, e não existe um "notebook de instalação".

REPO = 'https://github.com/FelypeSR/TCC_Cristian.git'   # ← ajuste aqui

!git clone -q {REPO} /content/TCC_Cristian 2>/dev/null || (cd /content/TCC_Cristian && git pull -q)
!pip install -q -r /content/TCC_Cristian/requirements.txt

from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/TCC_Cristian/src')

import config as CFG
CFG.fixar_seeds()
CFG.criar_pastas()
CFG.resumo()

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import preprocessing as pp
import evaluation as ev

# O modelo do spaCy precisa ser baixado a cada runtime do Colab
!python -m spacy download pt_core_news_sm -q

treino = pd.read_csv(CFG.SPLIT_FILES['train'], encoding='utf-8')
val    = pd.read_csv(CFG.SPLIT_FILES['val'],   encoding='utf-8')
teste  = pd.read_csv(CFG.SPLIT_FILES['test'],  encoding='utf-8')

print(f'Treino: {len(treino)}  |  Val: {len(val)}  |  Teste: {len(teste)}')

## 2. Pré-processamento`preparar_classico` aplica normalização, substituição por tokens especiais elematização com spaCy. A remoção de stopwords fica **desligada por padrão**: SMStêm 15 a 25 palavras, e marcadores de urgência e comando ("já", "agora", "você")estão na lista de stopwords do NLTK — são exatamente o vocabulário do golpe.A célula de ablação, ao final, compara as duas versões.

In [ ]:
USAR_STOPWORDS = False   # ver ablação na seção 8

X_treino_txt = pp.preparar_classico(treino[CFG.COL_TEXTO], stopwords=USAR_STOPWORDS)
X_val_txt    = pp.preparar_classico(val[CFG.COL_TEXTO],    stopwords=USAR_STOPWORDS)
X_teste_txt  = pp.preparar_classico(teste[CFG.COL_TEXTO],  stopwords=USAR_STOPWORDS)

y_treino = treino[CFG.COL_ROTULO].values
y_val    = val[CFG.COL_ROTULO].values
y_teste  = teste[CFG.COL_ROTULO].values

print('=== Exemplo: original vs. processado ===')
for i in range(3):
    print(f'\n  [{y_treino[i]}]')
    print(f'  orig: {treino[CFG.COL_TEXTO].iloc[i][:110]}')
    print(f'  proc: {X_treino_txt[i][:110]}')

## 3. VetorizaçãoTF-IDF com bigramas, mais as features artesanais (presença de URL, urgência,instituição financeira, promessa de prêmio, proporção de maiúsculas). Asfeatures artesanais são calculadas sobre o texto **original** — caixa alta epontos de exclamação se perdem na normalização.> **Cada modelo recebe uma matriz diferente, de propósito.** O Naive Bayes é um> modelo multinomial: ele pressupõe contagens/frequências de termos. Injetar> features contínuas escaladas (`n_chars`, `prop_maiusculas`) viola essa> premissa e, na prática, faz os pesos degenerarem — a lista de termos> discriminativos vira ruído. O SVM é linear e não tem essa restrição, então> aproveita as features artesanais normalmente.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from scipy.sparse import hstack, csr_matrix

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),   # bigramas capturam padrões como "clique aqui"
    max_features=20_000,
    sublinear_tf=True,    # log(1 + tf) reduz o peso de termos muito frequentes
    min_df=2,             # ignora termos que aparecem em um único documento
)

# fit apenas no treino; val e teste só são transformados
T_treino = vectorizer.fit_transform(X_treino_txt)
T_val    = vectorizer.transform(X_val_txt)
T_teste  = vectorizer.transform(X_teste_txt)

# MinMaxScaler, não StandardScaler: valores centrados em zero seriam negativos
escala = MinMaxScaler()
F_treino = escala.fit_transform(pp.matriz_features(treino[CFG.COL_TEXTO]))
F_val    = escala.transform(pp.matriz_features(val[CFG.COL_TEXTO]))
F_teste  = escala.transform(pp.matriz_features(teste[CFG.COL_TEXTO]))

# Naive Bayes: só TF-IDF (premissa multinomial)
# SVM: TF-IDF + features artesanais
MATRIZES = {
    'naive_bayes': (T_treino, T_val, T_teste),
    'svm': (
        hstack([T_treino, csr_matrix(F_treino)]).tocsr(),
        hstack([T_val,    csr_matrix(F_val)]).tocsr(),
        hstack([T_teste,  csr_matrix(F_teste)]).tocsr(),
    ),
}

print(f'Vocabulário     : {len(vectorizer.vocabulary_)} termos')
print(f'Naive Bayes     : {MATRIZES["naive_bayes"][0].shape}  (só TF-IDF)')
print(f'SVM             : {MATRIZES["svm"][0].shape}  (TF-IDF + {F_treino.shape[1]} artesanais)')

## 4. Treinamento

In [ ]:
from sklearn.naive_bayes import ComplementNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

# ComplementNB estima o complemento de cada classe — supera o MultinomialNB
# quando as classes são desbalanceadas, que é o caso aqui
cnb = ComplementNB(alpha=1.0)
cnb.fit(MATRIZES['naive_bayes'][0], y_treino)

# class_weight='balanced' compensa o desbalanceamento; sem isso o modelo
# tende a favorecer a classe majoritária, contrariando a priorização do recall.
# method='sigmoid': a calibração isotônica superajusta em corpus pequeno.
svm = CalibratedClassifierCV(
    LinearSVC(C=1.0, max_iter=5000, class_weight='balanced', random_state=CFG.SEED),
    cv=5, method='sigmoid',
)
svm.fit(MATRIZES['svm'][0], y_treino)

MODELOS = {'naive_bayes': cnb, 'svm': svm}
print('Modelos treinados.')

## 5. Calibração do limiar na validaçãoO limiar não fica em 0.5 por omissão. Como o falso negativo é o erro mais caro,o ponto de operação é uma decisão de projeto — e escolhê-lo na **validação**(nunca no teste) é o que a torna defensável.

In [ ]:
limiares = {}

for nome, modelo in MODELOS.items():
    _, Xva, _ = MATRIZES[nome]
    idx_pos = list(modelo.classes_).index(CFG.CLASSE_POSITIVA)
    score_val = modelo.predict_proba(Xva)[:, idx_pos]

    limiar, f2_val = ev.calibrar_limiar(y_val, score_val, beta=2)
    limiares[nome] = limiar

    m05 = ev.calcular_metricas(y_val, ev.aplicar_limiar(score_val, 0.5), score_val)
    mca = ev.calcular_metricas(y_val, ev.aplicar_limiar(score_val, limiar), score_val)

    print(f'\n{nome}  — limiar escolhido: {limiar:.4f}')
    print(f'  val @0.5       F2={m05["f2"]:.4f}  recall={m05["recall"]:.4f}  FN={m05["FN"]}')
    print(f'  val @calibrado F2={mca["f2"]:.4f}  recall={mca["recall"]:.4f}  FN={mca["FN"]}')

## 6. Avaliação no conjunto de testeO teste é tocado **uma única vez**, aqui, com o limiar já fixado na validação.

In [ ]:
resultados, predicoes_teste = {}, {}

for nome, modelo in MODELOS.items():
    _, _, Xte = MATRIZES[nome]
    idx_pos = list(modelo.classes_).index(CFG.CLASSE_POSITIVA)
    score = modelo.predict_proba(Xte)[:, idx_pos]
    pred = ev.aplicar_limiar(score, limiares[nome])

    resultados[nome] = ev.calcular_metricas(y_teste, pred, score)
    predicoes_teste[nome] = pred
    ev.salvar_predicoes(teste['id'], y_teste, pred, score, nome)

tabela = pd.DataFrame(resultados).T
print('\n=== Teste ===')
print(ev.formatar_tabela(tabela).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (nome, metricas) in zip(axes, resultados.items()):
    ev.plot_confusao(y_teste, predicoes_teste[nome],
                     f'{nome}\nF2={metricas["f2"]:.3f}  Recall={metricas["recall"]:.3f}', ax=ax)

plt.suptitle('Matrizes de confusão — baselines clássicos (conjunto de teste)')
plt.tight_layout()
plt.savefig(f"{CFG.PATHS['figures']}/02_confusao_baselines.png", dpi=150, bbox_inches='tight')
plt.show()

## 7. Termos mais discriminativosInsumo da análise qualitativa (seção 4.1) e defesa contra a pergunta "o que omodelo aprendeu?".> **Sobre a leitura do `ComplementNB`:** o nome `feature_log_prob_` sugere pesos> do complemento, o que levaria a inverter a ordenação. Não é o caso na> implementação do sklearn: com o padrão `norm=False`, o atributo guarda o> **negativo** do log das frequências do complemento, e a predição é feita por> `argmax`. Peso **alto** significa, portanto, termo mais associado à classe —> ordenação **decrescente**, igual ao `MultinomialNB`. A célula seguinte> confirma isso empiricamente; não altere a ordenação sem rodar essa checagem.

In [ ]:
# O ComplementNB foi treinado só com TF-IDF, então os nomes vêm apenas do
# vectorizer — é o que mantém esta análise legível.
nomes_features = np.array(vectorizer.get_feature_names_out())
N = 20

for i, classe in enumerate(cnb.classes_):
    # decrescente: no ComplementNB do sklearn (norm=False), peso ALTO = mais
    # associado à classe. Ver checagem na célula seguinte.
    top = np.argsort(cnb.feature_log_prob_[i])[::-1][:N]
    print(f'\nTop {N} termos de [{classe.upper()}]:')
    print('  ' + ' | '.join(nomes_features[top]))

In [ ]:
# Checagem da ordenação: em um corpus controlado, o ComplementNB deve produzir
# a mesma lista que o MultinomialNB, cuja leitura é inequívoca. Se estas duas
# listas divergirem, a ordenação acima está errada.
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

docs = ['golpe clique link banco urgente'] * 30 + ['bolo festa neta domingo receita'] * 30
rot = [CFG.CLASSE_POSITIVA] * 30 + [CFG.CLASSE_NEGATIVA] * 30

v_chk = CountVectorizer()
X_chk = v_chk.fit_transform(docs)
nomes_chk = np.array(v_chk.get_feature_names_out())

c_chk = ComplementNB().fit(X_chk, rot)
m_chk = MultinomialNB().fit(X_chk, rot)

for i, classe in enumerate(c_chk.classes_):
    top_c = set(nomes_chk[np.argsort(c_chk.feature_log_prob_[i])[::-1][:5]])
    top_m = set(nomes_chk[np.argsort(m_chk.feature_log_prob_[i])[::-1][:5]])
    print(f'  [{classe:9}] CNB == MNB: {"OK" if top_c == top_m else "DIVERGIU — inverta a ordenação"}')

## 8. Ablação: contribuição das mensagens sintéticasResponde numericamente à pergunta "a augmentation ajudou?", transformando umaescolha metodológica em resultado. Roda em CPU, custa minutos.

In [ ]:
if 'fonte' in treino.columns and (treino['fonte'] == 'sintetica').any():
    so_reais = treino[treino['fonte'] != 'sintetica'].reset_index(drop=True)

    # Mesmo pipeline do Naive Bayes acima: só TF-IDF, refeito sobre o
    # treino reduzido (refazer o fit é obrigatório — o vocabulário muda)
    Xr_txt = pp.preparar_classico(so_reais[CFG.COL_TEXTO], stopwords=USAR_STOPWORDS)
    vec_r = TfidfVectorizer(ngram_range=(1, 2), max_features=20_000, sublinear_tf=True, min_df=2)
    Xr = vec_r.fit_transform(Xr_txt)
    Xte_r = vec_r.transform(X_teste_txt)

    linhas = {}
    for rotulo, (X_fit, y_fit, X_ev) in {
        'com sintéticas': (MATRIZES['naive_bayes'][0], y_treino, MATRIZES['naive_bayes'][2]),
        'só reais':       (Xr, so_reais[CFG.COL_ROTULO].values, Xte_r),
    }.items():
        modelo = ComplementNB(alpha=1.0).fit(X_fit, y_fit)
        idx = list(modelo.classes_).index(CFG.CLASSE_POSITIVA)
        s = modelo.predict_proba(X_ev)[:, idx]
        linhas[rotulo] = ev.calcular_metricas(y_teste, ev.aplicar_limiar(s, 0.5), s)

    print('=== Ablação (Naive Bayes, limiar 0.5) ===')
    print(ev.formatar_tabela(pd.DataFrame(linhas).T).to_string())
    print(f"\nTreino com sintéticas: {len(treino)}  |  só reais: {len(so_reais)}")
else:
    print('[INFO] Sem mensagens sintéticas no treino — ablação não se aplica.')

## 9. Validação de pipeline (SMS Spam Collection)Confirma que a esteira TF-IDF → modelo → métricas funciona mecanicamente.**Este resultado não entra na comparação de modelos da monografia** e deveaparecer no texto claramente separado dos resultados de domínio.

In [ ]:
caminho_sms = f"{CFG.PATHS['raw']}/sms_spam_collection.csv"

if os.path.isfile(caminho_sms):
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import classification_report

    sms = pd.read_csv(caminho_sms, encoding='utf-8')
    s_tr, s_te = train_test_split(sms, test_size=0.15, random_state=CFG.SEED,
                                  stratify=sms[CFG.COL_ROTULO])

    v = TfidfVectorizer(max_features=5000)
    m = ComplementNB().fit(v.fit_transform(s_tr[CFG.COL_TEXTO]), s_tr[CFG.COL_ROTULO])

    print('=== Validação de pipeline — SMS Spam Collection (inglês) ===')
    print('(resultado mecânico, NÃO representa o domínio do TCC)\n')
    print(classification_report(s_te[CFG.COL_ROTULO],
                                m.predict(v.transform(s_te[CFG.COL_TEXTO])), zero_division=0))
else:
    print('[AVISO] sms_spam_collection.csv não encontrado — execute o notebook 01.')

print('\nProssiga para o notebook 03_bertimbau.ipynb.')